In [5]:
import pandas as pd
from datetime import datetime
import wm_tac_ops as ops
from ground_engine import GroundEngine, BattleContext

logs = pd.DataFrame(columns=[
    'current_time','initiator',
    'log_blue_id','log_blue_type','log_blue_inf_force','log_blue_arm_force',
    'log_blue_cas_inf','log_blue_cas_armor',
    'log_red_id','log_red_type','log_red_inf_force','log_red_arm_force',
    'log_red_cas_inf','log_red_cas_armor',
    'log_attack_type','log_result'
])

blue = ops.create_inf('blue_com', 100, 'blue')
red = ops.create_mech('red_inf', 50, 'red_arm',5, 'red')[0]

engine = GroundEngine()
context = BattleContext(
    attacker_cover=1,
    defender_cover=2,
    attacker_elevation=0,
    defender_elevation=0,
    distance=2,
    defender_returns_fire=True,
    attacker_berserk=False,
    defender_berserk=False,
)

result = engine.resolve_engagement(
    attacker=blue,
    defender=red,
    context=context,
    logs=logs,
    current_time=datetime(2026, 4, 19, 12, 15)
)
logs = result.logs


In [7]:
result = engine.resolve_engagement(
    attacker=blue,
    defender=red,
    context=context,
    logs=logs,
    current_time=datetime(2026, 4, 19, 12, 15)
)
logs = result.logs

In [8]:
logs

,current_time,initiator,log_blue_id,log_blue_type,log_blue_inf_force,log_blue_arm_force,log_blue_cas_inf,log_blue_cas_armor,log_red_id,log_red_type,log_red_inf_force,log_red_arm_force,log_red_cas_inf,log_red_cas_armor,log_attack_type,log_result
0,2026-04-19 12:00:00,blue,blue_com,inf,100,0,12,0,red_inf,mech,50,5,2,0,inf_attacks_mech,засада
1,2026-04-19 12:15:00,blue,blue_com,inf,88,0,0,0,red_inf,mech,48,5,14,3,inf_attacks_mech,разгром


In [38]:
result.logs

,current_time,initiator,log_blue_id,log_blue_type,log_blue_inf_force,log_blue_arm_force,log_blue_cas_inf,log_blue_cas_armor,log_red_id,log_red_type,log_red_inf_force,log_red_arm_force,log_red_cas_inf,log_red_cas_armor,log_attack_type,log_result
0,2026-04-19 12:00:00,blue,blue_plt,inf,21,0,2,0,red_plt,inf,25,0,0,0,inf_attacks_inf,победа
1,2026-04-19 12:00:00,blue,blue_plt,inf,21,0,0,0,red_plt,inf,22,0,3,0,inf_attacks_inf,разгром
2,2026-04-19 12:00:00,blue,blue_plt,inf,12,0,9,0,red_plt,inf,22,0,0,0,inf_attacks_inf,засада


In [36]:
logs

,current_time,initiator,log_blue_id,log_blue_type,log_blue_inf_force,log_blue_arm_force,log_blue_cas_inf,log_blue_cas_armor,log_red_id,log_red_type,log_red_inf_force,log_red_arm_force,log_red_cas_inf,log_red_cas_armor,log_attack_type,log_result
0,2026-04-19 12:00:00,blue,blue_plt,inf,21,0,2,0,red_plt,inf,25,0,0,0,inf_attacks_inf,победа
1,2026-04-19 12:00:00,blue,blue_plt,inf,21,0,0,0,red_plt,inf,22,0,3,0,inf_attacks_inf,разгром


In [2]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import pandas as pd


def add_hierarchy_uids(
    df: pd.DataFrame,
    battalion_col: str = "battalion_id",
    company_col: str = "company_id",
    platoon_col: str = "platoon_id",
    squad_col: str = "squad_id",
    soldier_col: str = "soldier_id",
) -> pd.DataFrame:
    """Return a copy of df with hierarchical unique IDs added.

    Expected raw template columns:
      - battalion_id
      - company_id
      - platoon_id
      - squad_id
      - soldier_id

    New columns:
      - battalion_uid: B1
      - company_uid: B1-C1
      - platoon_uid: B1-C1-P1
      - squad_uid: B1-C1-P1-S1
      - soldier_uid: B1-C1-P1-S1-U1

    Notes:
      - This keeps your original local IDs unchanged.
      - The generated UID columns are what the engine should use.
    """
    required = [battalion_col, company_col, platoon_col, squad_col, soldier_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    out = df.copy()

    # Keep source IDs as integers where possible so generated strings are clean.
    for col in required:
        out[col] = pd.to_numeric(out[col], errors="raise").astype("Int64")

    out["battalion_uid"] = "B" + out[battalion_col].astype(str)
    out["company_uid"] = (
        out["battalion_uid"]
        + "-C"
        + out[company_col].astype(str)
    )
    out["platoon_uid"] = (
        out["company_uid"]
        + "-P"
        + out[platoon_col].astype(str)
    )
    out["squad_uid"] = (
        out["platoon_uid"]
        + "-S"
        + out[squad_col].astype(str)
    )
    out["soldier_uid"] = (
        out["squad_uid"]
        + "-U"
        + out[soldier_col].astype(str)
    )

    return out


def process_workbook(input_path: str | Path, output_path: str | Path, sheet_name: str | None = None) -> None:
    input_path = Path(input_path)
    output_path = Path(output_path)

    xls = pd.ExcelFile(input_path)
    target_sheets: Iterable[str]
    if sheet_name is None:
        target_sheets = xls.sheet_names
    else:
        target_sheets = [sheet_name]

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for current_sheet in xls.sheet_names:
            df = pd.read_excel(input_path, sheet_name=current_sheet)
            if current_sheet in target_sheets:
                df = add_hierarchy_uids(df)
            df.to_excel(writer, sheet_name=current_sheet, index=False)


if __name__ == "__main__":
    src = Path("brigade_template.xlsx")
    dst = Path("brigade_template_with_uids.xlsx")
    process_workbook(src, dst)
    print(f"Saved: {dst}")


Saved: brigade_template_with_uids.xlsx


In [10]:
from __future__ import annotations

import random
from dataclasses import asdict
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

from force_manager import ForceManager
from ground_engine import GroundEngine, BattleContext


# ============================================================
# CONFIG
# ============================================================
PLAYER_FILE = "player_brigade.xlsx"
ENEMY_FILE = "enemy_brigade.xlsx"

# Какие подразделения отправляем на бой.
# level может быть: "battalion", "company", "platoon", "squad"
PLAYER_LEVEL = "battalion"
PLAYER_FORMATIONS = ["B1"]

ENEMY_LEVEL = "battalion"
ENEMY_FORMATIONS = ["B1"]

# Сколько временных ground-юнитов собрать на сторону
TARGET_UNITS_PER_SIDE = 6

# Диапазон количества боёв за один тактический прогон
MIN_BATTLES = 3
MAX_BATTLES = 15

# Диапазоны случайных параметров столкновения
DISTANCE_RANGE = (1, 3)
COVER_RANGE = (1, 2)
ELEVATION_RANGE = (0, 0)  # пока высоту не рандомим

# Сохранять результаты в новые файлы
PLAYER_OUTPUT_FILE = "player_brigade_after_battle.xlsx"
ENEMY_OUTPUT_FILE = "enemy_brigade_after_battle.xlsx"
LOG_OUTPUT_FILE = "tactical_battle_log.xlsx"

# Время старта тактического прогона
START_TIME = datetime(2026, 4, 19, 8, 0)
TIME_STEP_MINUTES = 15
# ============================================================


LOG_COLUMNS = [
    'current_time', 'initiator',
    'log_blue_id', 'log_blue_type', 'log_blue_inf_force', 'log_blue_arm_force',
    'log_blue_cas_inf', 'log_blue_cas_armor',
    'log_red_id', 'log_red_type', 'log_red_inf_force', 'log_red_arm_force',
    'log_red_cas_inf', 'log_red_cas_armor',
    'log_attack_type', 'log_result'
]


class TacticalTestRunner:
    def __init__(self, player_file: str, enemy_file: str):
        self.player_manager = ForceManager.from_excel(player_file)
        self.enemy_manager = ForceManager.from_excel(enemy_file)
        self.engine = GroundEngine()
        self.logs = pd.DataFrame(columns=LOG_COLUMNS)
        self.current_time = START_TIME

    @staticmethod
    def _alive_units(units):
        return [u for u in units if len(u.alive_df) > 0]

    @staticmethod
    def _choose_random_context() -> BattleContext:
        attacker_cover = random.randint(*COVER_RANGE)
        defender_cover = random.randint(*COVER_RANGE)
        attacker_elevation = random.randint(*ELEVATION_RANGE)
        defender_elevation = random.randint(*ELEVATION_RANGE)
        distance = random.randint(*DISTANCE_RANGE)

        # В большинстве столкновений предполагаем ответный огонь
        defender_returns_fire = True

        return BattleContext(
            attacker_cover=attacker_cover,
            defender_cover=defender_cover,
            attacker_elevation=attacker_elevation,
            defender_elevation=defender_elevation,
            distance=distance,
            defender_returns_fire=defender_returns_fire,
            attacker_berserk=False,
            defender_berserk=False,
        )

    def build_forces(self):
        player_units, player_plan = self.player_manager.generate_ground_units(
            level=PLAYER_LEVEL,
            formation_uids=PLAYER_FORMATIONS,
            target_units=TARGET_UNITS_PER_SIDE,
            side="blue",
            unit_prefix="BLU",
        )

        enemy_units, enemy_plan = self.enemy_manager.generate_ground_units(
            level=ENEMY_LEVEL,
            formation_uids=ENEMY_FORMATIONS,
            target_units=TARGET_UNITS_PER_SIDE,
            side="red",
            unit_prefix="RED",
        )

        return player_units, enemy_units, player_plan, enemy_plan

    def run(self):
        player_units, enemy_units, player_plan, enemy_plan = self.build_forces()

        battle_count = random.randint(MIN_BATTLES, MAX_BATTLES)
        tactical_summary = []

        for battle_no in range(1, battle_count + 1):
            alive_blue = self._alive_units(player_units)
            alive_red = self._alive_units(enemy_units)

            if not alive_blue or not alive_red:
                print("Одна из сторон больше не имеет боеспособных временных юнитов. Прогон завершён раньше.")
                break

            attacker = random.choice(alive_blue)
            defender = random.choice(alive_red)
            context = self._choose_random_context()

            result = self.engine.resolve_engagement(
                attacker=attacker,
                defender=defender,
                context=context,
                logs=self.logs,
                current_time=self.current_time,
            )

            self.logs = result.logs
            self.current_time += timedelta(minutes=TIME_STEP_MINUTES)

            last_log = result.logs.iloc[-1]

            tactical_summary.append({
                "battle_no": battle_no,
                "time": last_log["current_time"],
                "blue_unit": last_log["log_blue_id"],
                "red_unit": last_log["log_red_id"],
                "attack_type": last_log["log_attack_type"],
                "result": last_log["log_result"],
                "blue_force_before": last_log["log_blue_inf_force"],
                "blue_cas_inf": last_log["log_blue_cas_inf"],
                "blue_cas_armor": last_log["log_blue_cas_armor"],
                "red_force_before": last_log["log_red_inf_force"],
                "red_cas_inf": last_log["log_red_cas_inf"],
                "red_cas_armor": last_log["log_red_cas_armor"],
                "distance": context.distance,
                "blue_cover": context.attacker_cover,
                "red_cover": context.defender_cover,
            })

        # Применяем итоговые изменения временных юнитов обратно в master tables
        self.player_manager.apply_many_battle_results(player_units)
        self.enemy_manager.apply_many_battle_results(enemy_units)

        # Сохраняем обновлённые таблицы и лог
        self.player_manager.save_to_excel(PLAYER_OUTPUT_FILE)
        self.enemy_manager.save_to_excel(ENEMY_OUTPUT_FILE)

        summary_df = pd.DataFrame(tactical_summary)
        with pd.ExcelWriter(LOG_OUTPUT_FILE, engine="openpyxl") as writer:
            self.logs.to_excel(writer, sheet_name="battle_log", index=False)
            summary_df.to_excel(writer, sheet_name="summary", index=False)
            player_plan.to_excel(writer, sheet_name="player_plan", index=False)
            enemy_plan.to_excel(writer, sheet_name="enemy_plan", index=False)

        return {
            "battle_count": len(tactical_summary),
            "logs": self.logs,
            "summary": summary_df,
            "player_plan": player_plan,
            "enemy_plan": enemy_plan,
            "player_units": player_units,
            "enemy_units": enemy_units,
        }


def print_short_summary(result_bundle: dict):
    summary = result_bundle["summary"]
    print("=" * 60)
    print(f"Проведено боёв: {len(summary)}")
    print("=" * 60)

    if len(summary) == 0:
        print("Боёв не было.")
        return

    for _, row in summary.iterrows():
        blue_after = row["blue_force_before"] - row["blue_cas_inf"]
        red_after = row["red_force_before"] - row["red_cas_inf"]
        print(
            f"Бой #{row['battle_no']} | {row['time']} | "
            f"{row['blue_unit']} vs {row['red_unit']} | "
            f"dist={row['distance']} | cover {row['blue_cover']}:{row['red_cover']} | "
            f"{row['result']} | "
            f"blue {row['blue_force_before']} -> {blue_after} (-{row['blue_cas_inf']}) | "
            f"red {row['red_force_before']} -> {red_after} (-{row['red_cas_inf']})"
        )


if __name__ == "__main__":
    # Проверяем, что входные файлы существуют
    for file_path in [PLAYER_FILE, ENEMY_FILE]:
        if not Path(file_path).exists():
            raise FileNotFoundError(
                f"Не найден файл: {file_path}. Положи файлы рядом со скриптом или поправь CONFIG."
            )

    runner = TacticalTestRunner(PLAYER_FILE, ENEMY_FILE)
    result_bundle = runner.run()
    print_short_summary(result_bundle)
    print("\nФайлы сохранены:")
    print(f"- {PLAYER_OUTPUT_FILE}")
    print(f"- {ENEMY_OUTPUT_FILE}")
    print(f"- {LOG_OUTPUT_FILE}")


C:\Users\chaic\WMpython\WM_calculations\force_manager.py:363: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  master_index.loc[common_ids, "inf_kills"] = update_index.loc[common_ids, "inf_kills"]
C:\Users\chaic\WMpython\WM_calculations\force_manager.py:364: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  master_index.loc[common_ids, "apc_kills"] = update_index.loc[common_ids, "apc_kills"]
C:\Users\chaic\WMpyt

Проведено боёв: 15
Бой #1 | 2026-04-19 08:00:00 | BLU_5 vs RED_3 | dist=1 | cover 2:2 | ничья | blue 30 -> 29 (-1) | red 60 -> 60 (-0)
Бой #2 | 2026-04-19 08:15:00 | BLU_3 vs RED_3 | dist=1 | cover 1:1 | засада | blue 60 -> 43 (-17) | red 60 -> 59 (-1)
Бой #3 | 2026-04-19 08:30:00 | BLU_5 vs RED_4 | dist=3 | cover 1:2 | поражение | blue 29 -> 27 (-2) | red 30 -> 30 (-0)
Бой #4 | 2026-04-19 08:45:00 | BLU_5 vs RED_4 | dist=3 | cover 2:1 | поражение | blue 27 -> 27 (-0) | red 30 -> 26 (-4)
Бой #5 | 2026-04-19 09:00:00 | BLU_5 vs RED_4 | dist=2 | cover 2:1 | ничья | blue 27 -> 27 (-0) | red 26 -> 23 (-3)
Бой #6 | 2026-04-19 09:15:00 | BLU_6 vs RED_4 | dist=3 | cover 2:2 | разгром | blue 30 -> 30 (-0) | red 23 -> 16 (-7)
Бой #7 | 2026-04-19 09:30:00 | BLU_1 vs RED_6 | dist=3 | cover 1:1 | победа | blue 60 -> 58 (-2) | red 30 -> 30 (-0)
Бой #8 | 2026-04-19 09:45:00 | BLU_3 vs RED_4 | dist=1 | cover 1:2 | поражение | blue 43 -> 43 (-0) | red 16 -> 16 (-0)
Бой #9 | 2026-04-19 10:00:00 | BLU_2

In [12]:
player_manager

NameError: name 'player_manager' is not defined

In [11]:
PLAYER_OUTPUT_FILE

'player_brigade_after_battle.xlsx'